# Datenbereinigung mit Pandas – Schritt für Schritt (VHS)

**Ziel:** Wir lernen grundlegende Datenbereinigungsschritte, jeweils mit **Mini-Beispiel** und **Übung**.  
Am Ende bauen wir eine **fortgeschrittene Bereinigungspipeline** für einen realistisch „schmutzigen“ Datensatz.

> Notebook-Sprache: **Deutsch**  
> Empfohlen: Zellen von oben nach unten ausführen.


## 0) Setup + Mini-Datensatz (absichtlich „schmutzig“)


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

df = pd.DataFrame({
    "kunden_id": [101, 102, 103, 103, 104, None],
    "email": ["Alice@example.com ", "  ", None, "bob@EXAMPLE.com", "carla@example.com", "carla@example.com "],
    "age": ["29", " 34 ", "n/a", None, "-5", "120"],
    "signup_date": ["2025-01-05", "05.01.2025", "2025/02/01", None, "2025-13-01", "10-03-2025"],
    "country": ["de", "DE", " Germany", "tw", "TWN", "Taiwan"],
    "spend_eur": ["12,50", "10.00", "1.234,56", None, "-3", " 8,0 "],
    "source": ["Ads", "ads ", "Organic", "organic", "Referral", "referral"]
})

df


## 1) Schnell-Check: Überblick über Form, Typen, fehlende Werte, Duplikate


In [ ]:
print("Form (Zeilen, Spalten):", df.shape)
print("\nDatentypen:")
print(df.dtypes)

print("\nFehlende Werte pro Spalte:")
print(df.isna().sum())

print("\nDuplikate (ganze Zeilen):", df.duplicated().sum())

df.head()


### 🧩 Übung 1
- Welche Spalten enthalten fehlende Werte?
- Welche Spalten haben „komische“ Datentypen (z.B. Zahlen als Text)?


## 2) Spaltennamen sauber machen (Leerzeichen entfernen, Kleinschreibung, Unterstrich)


In [ ]:
df = df.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns


### 🧩 Übung 2
- Benenne eine Spalte um, z.B. `kunden_id` → `kunde_id` (mit `df.rename(...)`).


In [ ]:
# Beispiel: eine Spalte umbenennen
df_rename_demo = df.rename(columns={"kunden_id": "kunde_id"})
df_rename_demo.columns


## 3) Strings bereinigen + Groß/Kleinschreibung vereinheitlichen


In [ ]:
# Wir machen das für email, country, source
for col in ["email", "country", "source"]:
    df[col] = df[col].astype("string").str.strip()

df["email"] = df["email"].str.lower()
df["source"] = df["source"].str.lower()

df[["email", "country", "source"]]


### 🧩 Übung 3
- Vereinheitliche auch `country` auf lowercase (oder uppercase).
- Prüfe danach mit `df["country"].unique()` die Werte.


In [ ]:
df_ex3 = df.copy()
df_ex3["country"] = df_ex3["country"].astype("string").str.strip().str.lower()
df_ex3["country"].unique()


## 4) „Leere Strings“ (nur Leerzeichen) in echte fehlende Werte (NA) umwandeln


In [ ]:
# Regex: ^\s*$ bedeutet: nur Leerzeichen (oder leer)
df["email"] = df["email"].replace(r"^\s*$", pd.NA, regex=True)

df["email"]


### 🧩 Übung 4
- Wende dieselbe Logik auf `source` an (falls dort leere Werte vorkommen).


In [ ]:
df_ex4 = df.copy()
df_ex4["source"] = df_ex4["source"].replace(r"^\s*$", pd.NA, regex=True)
df_ex4["source"]


## 5) Duplikate entfernen (einfacher Einstieg)


In [ ]:
print("Vorher:", df.shape)
df_no_dupes = df.drop_duplicates()
print("Nachher:", df_no_dupes.shape)

df_no_dupes


### 🧩 Übung 5
- Prüfe Duplikate nur nach `email`:
  - `df.duplicated(subset=["email"])`
- Zeige alle Duplikat-Zeilen an.


In [ ]:
mask = df.duplicated(subset=["email"], keep=False)
df[mask].sort_values("email")


## 6) Zahlen konvertieren: Text → Zahl (Ungültiges wird NA)


In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="konvertieren")
df["age"]


### 🧩 Übung 6
- Wandle `kunden_id` in eine Zahl um (falls nötig) und prüfe danach `df.dtypes`.


In [ ]:
df_ex6 = df.copy()
df_ex6["kunden_id"] = pd.to_numeric(df_ex6["kunden_id"], errors="konvertieren")
df_ex6.dtypes


## 7) Plausibilitätsregeln: Werte außerhalb eines Bereichs → NA


In [ ]:
# Beispielregel: Alter muss zwischen 0 und 100 liegen
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = pd.NA
df["age"]


### 🧩 Übung 7
- Definiere eine Regel für `kunden_id`: z.B. darf nicht kleiner als 1 sein.


In [ ]:
df_ex7 = df.copy()
df_ex7.loc[df_ex7["kunden_id"].notna() & (df_ex7["kunden_id"] < 1), "kunden_id"] = pd.NA
df_ex7["kunden_id"]


## 8) Datum konvertieren: verschiedene Formate → datetime (Ungültiges wird NA)


In [ ]:
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="konvertieren", dayfirst=True)
df["signup_date"]


### 🧩 Übung 8
- Erstelle eine neue Spalte `signup_year` (Jahr aus dem Datum).


In [ ]:
df_ex8 = df.copy()
df_ex8["signup_year"] = df_ex8["signup_date"].dt.year
df_ex8[["signup_date", "signup_year"]]


## 9) Kategorien standardisieren: gleiche Bedeutung, gleiche Schreibweise


### Beispiel: Länder in einheitliche Codes (DE/TW) umwandeln


In [ ]:
# Erst normalisieren
df["country"] = df["country"].astype("string").str.strip().str.lower()

# Dann mappen
df["country"] = df["country"].replace({
    "de": "DE",
    "germany": "DE",
    "tw": "TW",
    "twn": "TW",
    "taiwan": "TW"
})

df["country"]


### 🧩 Übung 9
- Füge eine neue Regel hinzu: wenn `country` unbekannt ist, setze `country = "UNKNOWN"`.


In [ ]:
df_ex9 = df.copy()
df_ex9["country"] = df_ex9["country"].fillna("UNKNOWN")
df_ex9["country"]


## 10) Euro-Zahlenformate ohne Funktion (Schritt für Schritt)


Wir zeigen hier absichtlich **ohne Funktion** einen robusten Ansatz, um Mischformate zu parsen:

- `12,50` → 12.50  
- `1.234,56` → 1234.56  
- `10.00` → 10.00


In [ ]:
s = df["spend_eur"].astype("string").str.strip()

# Schritt 1: EU-Format erkennen (hat Komma UND Punkt)
mask = s.str.contains(",", na=False) & s.str.contains(r"\.", na=False)

# Schritt 2: EU-Format umwandeln: 1.234,56 -> 1234.56
s_eu = s.where(~mask, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

# Schritt 3: Rest: Komma als Dezimalpunkt behandeln
s_final = s_eu.str.replace(",", ".", regex=False)

# Schritt 4: zu Zahl
df["spend_eur"] = pd.to_numeric(s_final, errors="konvertieren")

df["spend_eur"]


### 🧩 Übung 10
- Setze negative Ausgaben (`spend_eur < 0`) auf NA.


In [ ]:
df.loc[df["spend_eur"] < 0, "spend_eur"] = pd.NA
df["spend_eur"]


## 11) fehlende Werte behandeln: löschen vs. auffüllen


### Option A: Zeilen löschen, wenn wichtige Werte fehlen


In [ ]:
df_drop = df.dropna(subset=["email"])
df_drop


### Option B: fehlende Werte auffüllen (Imputation) – einfache Variante


In [ ]:
df_fill = df.copy()
df_fill["spend_eur"] = df_fill["spend_eur"].fillna(0)
df_fill["age"] = df_fill["age"].fillna(df_fill["age"].median())

df_fill


### 🧩 Übung 11
- Entscheide selbst: Welche Spalten sind „kritisch“?
- Teste `dropna(subset=[...])` mit deiner Auswahl.


---


# Fortgeschritten: Bereinigungspipeline auf einem realistischeren Datensatz


In diesem Abschnitt kombinieren wir die Schritte und zeigen einen „professionelleren“ Arbeitsablauf:

- Daten-Check
- String-Normalisierung
- Typ-Konvertierung
- Regeln (Geschäftsregeln)
- Bericht am Ende


## A) Neuer Datensatz (noch etwas realistischer / mehr Chaos)


In [ ]:
df2 = pd.DataFrame({
    "Kunden-ID": [201, 202, 203, 203, 204, 205, None],
    "E-Mail": ["eva@Example.com ", "frank@sample.de", "  ", "frank@sample.de ", None, "gina@sample.de", "henry@sample.de"],
    "Alter": ["41", "unknown", " 39 ", "-2", None, "105", "33"],
    "Anmeldedatum": ["2025-04-01", "01.04.2025", "2025/04/03", "2025-04-03", "2025-04-31", None, "03-04-2025"],
    "Land": ["DE", " germany", "de", "tw", "TWN", "taiwan", None],
    "Ausgaben (€)": ["2.345,10", "19,99", "10.00", None, "-1", " 0 ", "7,5"],
    "Quelle": ["Ads", "Referral", "Organic", "organic", "ads ", None, "Referral"]
})

df2


## B) Pipeline: alles in klaren, gut erklärbaren Schritten


In [ ]:
df2_clean = df2.copy()

# 1) Spaltennamen vereinheitlichen
df2_clean.columns = (
    df2_clean.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("ä", "ae")
      .str.replace("ö", "oe")
      .str.replace("ü", "ue")
)

# 2) Strings bereinigen + casing
for col in ["e-mail", "land", "quelle"]:
    df2_clean[col] = df2_clean[col].astype("string").str.strip()

df2_clean["e-mail"] = df2_clean["e-mail"].str.lower()
df2_clean["quelle"] = df2_clean["quelle"].str.lower()

# 3) Leere Strings -> NA
for col in ["e-mail", "quelle", "land"]:
    df2_clean[col] = df2_clean[col].replace(r"^\s*$", pd.NA, regex=True)

# 4) Duplikate entfernen (ganze Zeilen)
df2_clean = df2_clean.drop_duplicates()

# 5) Zahlen: Alter
df2_clean["alter"] = pd.to_numeric(df2_clean["alter"], errors="konvertieren")
df2_clean.loc[(df2_clean["alter"] < 0) | (df2_clean["alter"] > 100), "alter"] = pd.NA

# 6) Datum
df2_clean["anmeldedatum"] = pd.to_datetime(df2_clean["anmeldedatum"], errors="konvertieren", dayfirst=True)

# 7) Land standardisieren
df2_clean["land"] = df2_clean["land"].astype("string").str.strip().str.lower().replace({
    "de": "DE",
    "germany": "DE",
    "tw": "TW",
    "twn": "TW",
    "taiwan": "TW"
})

# 8) Ausgaben: Euro-Parsing (hier kompakt, aber transparent)
s = df2_clean["ausgaben_(€)"].astype("string").str.strip()
mask = s.str.contains(",", na=False) & s.str.contains(r"\.", na=False)
s = s.where(~mask, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
s = s.str.replace(",", ".", regex=False)
df2_clean["ausgaben_eur"] = pd.to_numeric(s, errors="konvertieren")
df2_clean.loc[df2_clean["ausgaben_eur"] < 0, "ausgaben_eur"] = pd.NA

# 9) Beispiel: kritische Felder
#    -> Zeilen entfernen, wenn E-Mail fehlt
df2_clean = df2_clean.dropna(subset=["e-mail"])

df2_clean


## C) Abschluss-Bericht: Was ist jetzt besser?


In [ ]:
print("Form vorher:", df2.shape)
print("Form nachher:", df2_clean.shape)

print("\nFehlende Werte nach Bereinigung:")
print(df2_clean.isna().sum())

print("\nDatentypen nach Bereinigung:")
print(df2_clean.dtypes)

df2_clean.head()


# Checkliste (Merken!)
- **Prüfung:** shape, dtypes, missing, duplicates  
- **Strings:** Leerzeichen entfernen + Groß-/Kleinschreibung vereinheitlichen  
- **Fehlende Werte:** leere Strings → NA  
- **Typen:** Zahlen/Datum korrekt konvertieren  
- **Kategorien:** Mapping/Standardisierung  
- **Regeln:** Range, negative Werte  
- **Final:** Bericht + Dokumentation
